In [1]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]



if "mimi" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"
if "ivan" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"

INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)



mimi.uio.no


## Import python packages

* `Python` environment requirements: file requirements_retrieval.txt
* load `python` packages from imports.py
* load `functions` from functions.py



In [2]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(xr, glob, plt, np, DateFormatter, datetime, cm, pd, da, IMProToo,LinearSegmentedColormap, BoundaryNorm, date2num,time)
# from matplotlib.collections import PathCollection
import functions as fct
version = IMProToo.__version__


In [3]:
# reload imports
%load_ext autoreload
%autoreload 2

In [4]:
site = "ANX"
year = "2023"

In [5]:
met_wd    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_from_direction_{year}*.nc')))
met_ws    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_speed_{year}*.nc')))

In [6]:
ds_met_wd     = fct.open_or_empty(met_wd, decode_times=True, preprocess=lambda x: x.sortby('time'))
ds_met_ws     = fct.open_or_empty(met_ws, decode_times=True, preprocess=lambda x: x.sortby('time'))

In [7]:
_wd = ds_met_wd['wind_from_direction']
_ws = ds_met_ws['wind_speed']

In [8]:

_time = pd.date_range(start="2023-02-01", end="2023-03-30", freq='min')

In [9]:
# create new dataset with NaNs and time resolution 1 minute
dsx = xr.Dataset(data_vars=dict(
            wind_direction  =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            wind_speed      =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
        ),
                 coords=dict(
                     time       =_time,))

dsx['wind_direction']   = fct.arrays_same_time_res(_wd, dsx['wind_direction'])
dsx['wind_speed']       = fct.arrays_same_time_res(_ws, dsx['wind_speed'])

In [10]:
if len(_wd) != 0:
    dsx['wind_direction_filled'] = fct.fill_10min_to_1min(dsx, 'wind_direction')
if len(_ws) != 0:
    dsx['wind_speed_filled'] = fct.fill_10min_to_1min(dsx, 'wind_speed')

In [20]:
(dsx[['wind_direction_filled', 'wind_speed_filled']]).to_netcdf((os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/ANX_wind_10min_to_1min_filled_{year}.nc')))